# <font color="#418FDE" size="6.5" uppercase>**Privacy and Privacy-Enhancing Technologies**</font>

>Last update: 20250209.
    
By the end of this Lecture, you will be able to:
- Explain privacy risks associated with AI systems.
- Implement basic privacy-enhancing techniques.
- Evaluate trade-offs between privacy, model performance, and user trust.


## **1. Understanding Privacy Threats in AI: Data Breaches, Re-identification, and Inference Attacks**

### **1.1. De-Anonymization Tactics: Unmasking Hidden Identities**

<img src="https://storage.googleapis.com/mrafiei-odu/module_07/Lecture%20B/image_01_01.png" width="250">



>* De-anonymization threatens individual data anonymity.
* Auxiliary data sources increase re-identification risk.

>* Exploits weaknesses in anonymization techniques.
* Breaches privacy, poses ethical and legal challenges.

>* Implement robust privacy-preserving techniques beyond anonymization.
* Balance data utility with privacy protection.



### **1.2. Understanding Model Inversion and Membership Inference Risks**

<img src="https://storage.googleapis.com/mrafiei-odu/module_07/Lecture%20B/image_01_02.png" width="250">



>* Model inversion reveals sensitive input data.
* Public models increase privacy breach risks.

>* Membership inference attacks target training data inclusion.
* Sensitive datasets require robust privacy techniques.

>* Differential privacy adds noise to outputs.
* Federated learning trains models on devices.



In [ ]:
#@title Python Code - Understanding Model Inversion and Membership Inference Risks

# Demonstration of Privacy Attacks on Machine Learning Models:
# 1. Membership Inference Attack: Infers if a given sample was used during training,
#    based on the model's prediction confidence.
# 2. Model Inversion Attack: Attempts to reconstruct an input that the model associates
#    with a specific target class by optimizing an input image.
# Note: These examples are simplified and for educational purposes only.

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from sklearn.model_selection import train_test_split

# -------------------------
# Step 1: Load and Prepare Data
# -------------------------
# Load the MNIST dataset, which contains 28x28 grayscale images of handwritten digits.
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize the pixel values to the [0, 1] range for better training performance.
x_train, x_test = x_train / 255.0, x_test / 255.0

# For the purpose of demonstrating a membership inference attack,
# split the training data into two sets:
# - x_train_main: used to actually train the model.
# - x_shadow: acts as a "shadow" set (data NOT seen during training).
x_train_main, x_shadow, y_train_main, y_shadow = train_test_split(
    x_train, y_train, test_size=0.5, random_state=42
)

# -------------------------
# Step 2: Define and Train a Simple Model
# -------------------------
# Here we build a simple neural network model with one hidden layer.
model = Sequential([
    Flatten(input_shape=(28, 28)),   # Flatten the 2D image into a 1D vector.
    Dense(128, activation='relu'),   # Hidden layer with 128 neurons and ReLU activation.
    Dense(10, activation='softmax')    # Output layer for 10 classes with softmax activation.
])

# Compile the model with the Adam optimizer and sparse categorical crossentropy loss.
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Train the model for 5 epochs on the main training set.
print("Training the model on the main training set...")
model.fit(x_train_main, y_train_main, epochs=5, verbose=2)

# -------------------------
# Step 3: Membership Inference Attack
# -------------------------
# In a membership inference attack, an attacker tries to determine whether a particular
# data sample was used during training. Often, the model will output higher confidence
# for training samples, especially if the model is overfitted.
def membership_inference_attack(model, sample, threshold=0.9):
    """
    Determines if a sample is likely to have been in the training set based on the model's
    output confidence.
    Args:
      model: The trained neural network.
      sample: A single input sample (e.g., an image).
      threshold: Confidence threshold; if the maximum predicted probability exceeds this,
                 we infer that the sample was likely part of the training set.
    Returns:
      Boolean: True if the sample is inferred to be in the training set, False otherwise.
    """
    # Obtain the prediction probabilities for the input sample.
    prediction = model.predict(np.array([sample]))[0]
    # If the highest probability exceeds the threshold, assume the sample was used during training.
    return np.max(prediction) > threshold

# Pick one sample from the shadow set (which was NOT used during training).
shadow_sample = x_shadow[0]

# Apply the membership inference attack to the selected shadow sample.
is_member = membership_inference_attack(model, shadow_sample)

# Output the result of the membership inference attack.
print("\nMembership Inference Attack Result:")
if is_member:
    print("The sample is predicted to be a member of the training set.")
else:
    print("The sample is NOT predicted to be a member of the training set.")

# -------------------------
# Step 4: Model Inversion Attack
# -------------------------
# In a model inversion attack, an attacker attempts to reconstruct an input that the model
# associates with a given output. Here, we try to generate an image that the model will classify
# as a target class. This demonstrates how an adversary might extract information that could
# reveal sensitive details about the training data.
# In our example, we choose a target class (e.g., digit '3') and start with a random image.
# We then use gradient ascent to adjust the image so that the model's predicted probability for
# the target class increases.
target_class = 3  # Define the target class for which we want to generate an input.

# Initialize a random image (starting point for inversion) as a TensorFlow variable.
# The image is of shape (1, 28, 28) with pixel values in the [0, 1] range.
inversion_image = tf.Variable(tf.random.uniform((1, 28, 28), minval=0, maxval=1), trainable=True)

# Set up an optimizer for the inversion attack. We'll use Adam with a relatively high learning rate.
optimizer = tf.optimizers.Adam(learning_rate=0.1)

# Set the number of iterations for the optimization loop.
num_iterations = 300

print("\nPerforming Model Inversion Attack to generate an image for class", target_class, "...")
for i in range(num_iterations):
    with tf.GradientTape() as tape:
        # Ensure that the inversion_image is being tracked for gradient computations.
        tape.watch(inversion_image)
        # Forward pass: get the model's predictions for the current inversion_image.
        predictions = model(inversion_image)
        # Extract the probability of the target class.
        target_prob = predictions[0, target_class]
        # Define the loss as the negative log probability of the target class.
        # Minimizing this loss will maximize the probability for the target class.
        loss = -tf.math.log(target_prob + 1e-10)  # Add epsilon to avoid log(0).
        # (Optional) Add a regularization term to discourage extreme pixel values.
        regularization = 0.0001 * tf.reduce_sum(tf.square(inversion_image))
        total_loss = loss + regularization
    # Compute the gradient of the loss with respect to the inversion_image.
    gradients = tape.gradient(total_loss, inversion_image)
    # Apply the gradients to update the inversion_image.
    optimizer.apply_gradients([(gradients, inversion_image)])
    # Ensure that the updated inversion_image values remain within [0, 1].
    inversion_image.assign(tf.clip_by_value(inversion_image, 0.0, 1.0))
    # Optionally, print the loss every 50 iterations to monitor progress.
    if (i + 1) % 50 == 0:
        print(f"Iteration {i+1}, Loss: {total_loss.numpy()}")

# After the optimization loop, inversion_image should now resemble an image that the model
# associates with the target class.
# Convert the tensor to a NumPy array and remove the batch dimension for visualization.
inverted_img_np = inversion_image.numpy().squeeze()

# Display the generated (inverted) image using matplotlib.
plt.figure(figsize=(3, 3))
plt.title(f"Model Inversion Attack: Generated Image for Class {target_class}")
plt.imshow(inverted_img_np, cmap='gray')
plt.axis('off')
plt.show()

print("\nDemo completed: Both Membership Inference and Model Inversion Attacks have been showcased.")


### **1.3. Impact on Individuals and Organizations**

<img src="https://storage.googleapis.com/mrafiei-odu/module_07/Lecture%20B/image_01_03.png" width="250">



>* AI breaches risk identity theft and distress.
* Privacy loss hinders AI data sharing innovation.

>* Data breaches harm trust and incur penalties.
* Balancing AI benefits with privacy compliance challenges.

>* Privacy erosion fosters surveillance culture, stifles expression.
* Marginalized groups face increased discrimination, social injustice.



### **1.4. Navigating Legal Frameworks and Compliance**

<img src="https://storage.googleapis.com/mrafiei-odu/module_07/Lecture%20B/image_01_04.png" width="250">



>* GDPR mandates data protection by design.
* Non-compliance leads to fines and reputational damage.

>* CCPA grants data rights to Californians.
* AI regulations adapt to emerging privacy challenges.

>* Anticipate and prepare for regulatory changes.
* Engage with policymakers to shape AI regulations.



### **1.5. Promoting Openness and Accountability in AI Systems**

<img src="https://storage.googleapis.com/mrafiei-odu/module_07/Lecture%20B/image_01_05.png" width="250">



>* Transparency in AI models and data usage.
* Open-source community fosters AI innovation.

>* Trace decisions to ensure AI accountability.
* GDPR mandates transparency in automated decisions.

>* Partnership on AI fosters best practices.
* Google, Microsoft emphasize AI ethics guidelines.



## **2. Introduction to Privacy-Enhancing Technologies**

### **2.1. Decentralized Data Training with Federated Learning**

<img src="https://storage.googleapis.com/mrafiei-odu/module_07/Lecture%20B/image_02_01.png" width="250">



>* Federated Learning decentralizes data training for privacy.
* Google uses FL in Gboard for privacy.

>* Enhances privacy and promotes model inclusivity.
* Reduces biases in healthcare data training.

>* Challenges: model accuracy, communication overhead.
* Solutions: model compression, secure aggregation protocols.



### **2.2. Practical Steps for Differential Privacy Integration**

<img src="https://storage.googleapis.com/mrafiei-odu/module_07/Lecture%20B/image_02_02.png" width="250">



>* Differential privacy ensures individual data privacy.
* Privacy budget balances utility and privacy.

>* Select appropriate differential privacy mechanism.
* Laplace for sums, Gaussian for averages.

>* Regular audits ensure privacy and data utility.
* Educate stakeholders to foster trust and transparency.



### **2.3. Optimizing Trade-offs in Privacy and Utility**

<img src="https://storage.googleapis.com/mrafiei-odu/module_07/Lecture%20B/image_02_03.png" width="250">



>* Balancing privacy and utility is challenging.
* Differential privacy impacts model accuracy in healthcare.

>* Strategic approach evaluates AI needs and constraints.
* Hybrid approach balances privacy, utility, and trust.

>* Adaptive frameworks adjust privacy based on context.
* DECODE initiative empowers data control for citizens.



### **2.4. Integrating Diverse PETs for Enhanced Security Layers**

<img src="https://storage.googleapis.com/mrafiei-odu/module_07/Lecture%20B/image_02_04.png" width="250">



>* Combine PETs for comprehensive privacy protection.
* Federated learning and encryption prevent data breaches.

>* Adaptive security systems with diverse PETs.
* Blockchain ensures transparency and regulatory compliance.

>* Balance privacy, performance, and user experience.
* Google's federated learning maintains privacy and efficiency.



In [ ]:
#@title Python Code - Integrating Diverse PETs for Enhanced Security Layers

# This code demonstrates the integration of Federated Learning and Differential Privacy
# using a simple neural network model on the MNIST dataset. The purpose is to illustrate
# how these Privacy-Enhancing Technologies (PETs) can be combined to enhance security
# layers in AI systems. Federated Learning allows training on decentralized data, while
# Differential Privacy ensures that individual data points cannot be reverse-engineered.
# Note: This code is a simplified demonstration and does not implement full-fledged
# Federated Learning or Differential Privacy as would be done in a production environment.
# Libraries like TensorFlow Federated and PySyft can be used for more comprehensive solutions.

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

# Load the MNIST dataset
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Normalize the data
x_train, x_test = x_train / 255.0, x_test / 255.0

# Define a simple neural network model
model = keras.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

# Compile the model
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Simulate Federated Learning by splitting the training data into multiple clients
num_clients = 5
client_data = np.array_split(x_train, num_clients)
client_labels = np.array_split(y_train, num_clients)

# Train the model on each client separately (Federated Learning simulation)
for client in range(num_clients):
    print(f"Training on client {client+1}")
    model.fit(client_data[client], client_labels[client], epochs=1, verbose=1)

# Implement Differential Privacy by adding noise to the model weights
# This is a simplified example of adding noise to demonstrate the concept
for layer in model.layers:
    if hasattr(layer, 'kernel'):
        noise = np.random.normal(0, 0.1, layer.kernel.shape)
        layer.kernel.assign_add(noise)

# Evaluate the model on the test data
loss, accuracy = model.evaluate(x_test, y_test, verbose=2)
print(f"Test accuracy: {accuracy}")


### **2.5. Real-World Applications and Successes**

<img src="https://storage.googleapis.com/mrafiei-odu/module_07/Lecture%20B/image_02_05.png" width="250">



>* Federated learning protects patient privacy in healthcare.
* Enables collaborative research without data breaches.

>* Differential privacy protects user data in finance.
* Apple uses differential privacy for iOS features.

>* Smart cities use PETs for data security.
* Enhance services without compromising citizen privacy.



## **3. Balancing Privacy, Utility, and Trust in AI Systems**

### **3.1. Quantifying Performance Trade-offs in Privacy Models**

<img src="https://storage.googleapis.com/mrafiei-odu/module_07/Lecture%20B/image_03_01.png" width="250">



>* Balancing data protection with algorithm performance.
* Evaluate privacy needs against performance requirements.

>* Healthcare privacy impacts model accuracy.
* Quantify privacy impact for informed decisions.

>* Privacy and performance involve ethical considerations.
* Balance privacy with AI system effectiveness.



### **3.2. Integrating Privacy Strategies with Business Goals**

<img src="https://storage.googleapis.com/mrafiei-odu/module_07/Lecture%20B/image_03_02.png" width="250">



>* Privacy as a strategic business component.
* Enhances trust, loyalty, and competitive advantage.

>* Adopt holistic data lifecycle privacy approach.
* Align privacy strategies with business objectives.

>* Foster privacy culture through training programs.
* Prioritize privacy in decision-making processes.



### **3.3. Effective Stakeholder Communication on Privacy**

<img src="https://storage.googleapis.com/mrafiei-odu/module_07/Lecture%20B/image_03_03.png" width="250">



>* Align stakeholders with privacy goals and implications.
* Use workshops to bridge privacy understanding gaps.

>* Engage stakeholders in privacy decision-making process.
* Communicate privacy policies clearly to end-users.

>* Adapt messages for technical and non-technical audiences.
* Use dashboards to simplify complex privacy concepts.



### **3.4. Evolving Privacy Controls for AI Systems**

<img src="https://storage.googleapis.com/mrafiei-odu/module_07/Lecture%20B/image_03_04.png" width="250">



>* AI systems need robust privacy controls.
* Differential privacy protects data, ensures compliance.

>* Federated learning trains models without data exchange.
* Google uses federated learning for Android text.

>* Integrate privacy by design from development start.
* Homomorphic encryption enables secure data processing.



In [ ]:
#@title Python Code - Evolving Privacy Controls for AI Systems

# This Python code demonstrates the concept of federated learning using a simple example.
# Federated learning allows training a machine learning model across multiple devices
# without sharing the actual data, thus preserving privacy. In this example, we simulate
# federated learning using the MNIST dataset, where each client (device) trains a model
# on its local data and only shares the model updates with a central server.

# Import necessary libraries
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split

# Load the MNIST dataset
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Preprocess the data
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

# Split the training data into multiple clients (simulating different devices)
num_clients = 5
client_data = np.array_split(x_train, num_clients)
client_labels = np.array_split(y_train, num_clients)

# Define a simple CNN model
def create_model():
    model = keras.Sequential([
        layers.Conv2D(32, kernel_size=(3, 3), activation="relu", input_shape=(28, 28, 1)),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(10, activation="softmax")
    ])
    model.compile(optimizer="adam",
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

# Simulate federated learning
global_model = create_model()

# Train each client model locally and update the global model
for round in range(3):  # Simulate 3 rounds of federated learning
    print(f"Round {round + 1}")
    client_weights = []
    for client in range(num_clients):
        print(f"  Training client {client + 1}")
        client_model = create_model()
        client_model.set_weights(global_model.get_weights())
        client_model.fit(client_data[client], client_labels[client], epochs=1, verbose=0)
        client_weights.append(client_model.get_weights())
    # Average the weights from all clients to update the global model
    new_weights = [np.mean([client_weights[i][layer] for i in range(num_clients)], axis=0)
                   for layer in range(len(client_weights[0]))]
    global_model.set_weights(new_weights)

# Evaluate the global model on the test set
loss, accuracy = global_model.evaluate(x_test, y_test)
print(f"Test accuracy after federated learning: {accuracy:.4f}")


### **3.5. Emerging Trends and Innovations in Privacy-First AI**

<img src="https://storage.googleapis.com/mrafiei-odu/module_07/Lecture%20B/image_03_05.png" width="250">



>* Federated learning enhances privacy by localizing data.
* Google's Gboard uses federated learning for privacy.

>* Differential privacy ensures individual data point privacy.
* Apple uses differential privacy in iOS.

>* PETs enable secure computations on encrypted data.
* Advancements align AI with privacy expectations.



# <font color="#418FDE" size="6.5" uppercase>**Privacy and Privacy-Enhancing Technologies**</font>


In this lecture, you learned to:
- Explain privacy risks associated with AI systems.
- Implement basic privacy-enhancing techniques.
- Evaluate trade-offs between privacy, model performance, and user trust.

In the next Module (Module 8), we will go over 'Regulations, Standards, Certification, and AI Governance, Risk, and Compliance'